# Task 4 - Sobel Edge Detection using CUDA across many PNG images

**6CS005 High Performance Computing | Arkhan Shimar**

Each input produces three GPU outputs: the X-direction gradient, Y-direction gradient and combined Sobel edges. Together with the original these form four views of the same image. Zero padding preserves image dimensions. Directional views use clipped absolute gradients; the combined magnitude uses the original signed gradients.


## Get the public project files

The exact source revision is pinned so later repository edits cannot silently change these inputs.

In [ ]:
%%bash
set -euo pipefail
base="https://raw.githubusercontent.com/ArkhanShimar/HPC/b22f2d4676028e8e9f66d9c4aa74b010869b8cc5"
mkdir -p /content/HPC
cd /content/HPC
for file in Task4/sobel.cu Task4/lodepng.c Task4/lodepng.h Task4/LODEPNG_LICENSE data/images/flower.png data/images/4x4.png data/images/6x4.png data/images/sobel_target.png data/images/colour_ramp.png tests/verify.py tools/make_inputs.py; do
    mkdir -p "$(dirname "$file")"
    wget --quiet --tries=3 "$base/$file" -O "$file"
done


In [ ]:
%cd /content/HPC/Task4


## Runtime details

In [ ]:
%%bash
set -euo pipefail
{
date -u
gcc --version | head -1
lscpu | grep -E "Model name|CPU\(s\)|Thread|Core"
nvidia-smi
nvcc --version
} | tee environment.txt

## CUDA source

In [ ]:
%%writefile sobel.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>
#include <math.h>
#include <cuda_runtime.h>
extern "C" {
#include "lodepng.h"
}

__device__ int grey(const unsigned char *image, long long x, long long y, unsigned width, unsigned height) {
    if (x < 0 || y < 0 || x >= width || y >= height) return 0; // Zero padding.
    size_t p = ((size_t)y * width + (size_t)x)*4;
    return (77*(int)image[p] + 150*(int)image[p+1] + 29*(int)image[p+2]) >> 8;
}
__global__ void sobel_edges(const unsigned char *input, unsigned char *output, unsigned width, unsigned height) {
    size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= (size_t)width*height) return;
    long long x = i % width, y = i / width;
    const int gx[9] = {-1,0,1,-2,0,2,-1,0,1};
    const int gy[9] = {-1,-2,-1,0,0,0,1,2,1};
    int sx = 0, sy = 0;
    for (int dy = -1; dy <= 1; ++dy) {
        for (int dx = -1; dx <= 1; ++dx) {
            int value = grey(input, x+dx, y+dy, width, height);
            int k = (dy+1)*3+dx+1; sx += value*gx[k]; sy += value*gy[k];
        }
    }
    int magnitude = (int)(sqrtf((float)(sx*sx+sy*sy)) + 0.5f);
    int values[3] = {sx < 0 ? -sx : sx, sy < 0 ? -sy : sy, magnitude};
    size_t plane_bytes = (size_t)width*height*4;
    // Keep signed gradients for the magnitude; visualise their absolute values.
    for (int plane = 0; plane < 3; ++plane) {
        unsigned char value = (unsigned char)(values[plane] > 255 ? 255 : values[plane]);
        size_t p = (size_t)plane*plane_bytes+i*4;
        output[p] = output[p+1] = output[p+2] = value;
        output[p+3] = 255;
    }
}
static int checked(cudaError_t code, const char *action) {
    if (code == cudaSuccess) return 1;
    fprintf(stderr, "%s: %s\n", action, cudaGetErrorString(code)); return 0;
}
static int process_image(const char *filename) {
    unsigned char *input = NULL, *output = NULL, *device_in = NULL, *device_out = NULL;
    unsigned width = 0, height = 0, error;
    size_t pixels = 0, bytes = 0, blocks = 0;
    cudaEvent_t start = NULL, stop = NULL; cudaDeviceProp properties;
    int status = 1; float ms = 0;
    const char *suffixes[3] = {"_gx.png", "_gy.png", "_edges.png"};
    const char *base = strrchr(filename, '/'); base = base ? base+1 : filename;
    size_t name_length = strlen(base); char *out_name = NULL;
    if (name_length < 5 || strcmp(base+name_length-4, ".png")) { fprintf(stderr, "Expected a .png filename: %s\n", filename); return 1; }
    out_name = (char *)malloc(name_length+7);
    if (!out_name) return 1;
    memcpy(out_name, base, name_length-4); strcpy(out_name+name_length-4, "_edges.png");
    error = lodepng_decode32_file(&input, &width, &height, filename);
    if (error) { fprintf(stderr, "%s: %s\n", filename, lodepng_error_text(error)); goto cleanup; }
    if (!width || !height || (size_t)width > SIZE_MAX/(size_t)height/12) { fprintf(stderr, "Invalid image dimensions.\n"); goto cleanup; }
    pixels = (size_t)width*height; bytes = pixels*4; blocks = (pixels+255)/256;
    output = (unsigned char *)malloc(bytes*3);
    if (!output) goto cleanup;
    if (!checked(cudaGetDeviceProperties(&properties, 0), "GPU properties")) goto cleanup;
    if (blocks > (size_t)properties.maxGridSize[0]) { fprintf(stderr, "Image exceeds grid limit.\n"); goto cleanup; }
    if (!checked(cudaMalloc((void **)&device_in, bytes), "Allocate input image") ||
        !checked(cudaMalloc((void **)&device_out, bytes*3), "Allocate three output images") ||
        !checked(cudaMemcpy(device_in, input, bytes, cudaMemcpyHostToDevice), "Copy image") ||
        !checked(cudaEventCreate(&start), "Create start event") ||
        !checked(cudaEventCreate(&stop), "Create stop event") ||
        !checked(cudaEventRecord(start), "Record start")) goto cleanup;
    sobel_edges<<<(unsigned)blocks, 256>>>(device_in, device_out, width, height);
    if (!checked(cudaGetLastError(), "Launch Sobel kernel") ||
        !checked(cudaEventRecord(stop), "Record stop") ||
        !checked(cudaEventSynchronize(stop), "Wait for Sobel kernel") ||
        !checked(cudaEventElapsedTime(&ms, start, stop), "Measure kernel") ||
        !checked(cudaMemcpy(output, device_out, bytes*3, cudaMemcpyDeviceToHost), "Copy gradient and edge maps")) goto cleanup;
    for (int plane = 0; plane < 3; ++plane) {
        strcpy(out_name+name_length-4, suffixes[plane]);
        error = lodepng_encode32_file(out_name, output+(size_t)plane*bytes, width, height);
        if (error) { fprintf(stderr, "%s: %s\n", out_name, lodepng_error_text(error)); goto cleanup; }
        printf("Saved %s\n", out_name);
    }
    printf("%s -> %s | %ux%u | %zu blocks x 256 | Kernel %.3f ms\n", filename, out_name, width, height, blocks, ms);
    status = 0;
cleanup:
    if (start && !checked(cudaEventDestroy(start), "Destroy start event")) status = 1;
    if (stop && !checked(cudaEventDestroy(stop), "Destroy stop event")) status = 1;
    if (device_in && !checked(cudaFree(device_in), "Free device input")) status = 1;
    if (device_out && !checked(cudaFree(device_out), "Free device output")) status = 1;
    free(input); free(output); free(out_name); return status;
}
int main(int argc, char **argv) {
    if (argc < 2) { fprintf(stderr, "Usage: %s image1.png [image2.png ...]\n", argv[0]); return 1; }
    // Output names use input basenames. Reject collisions before processing.
    for (int i = 1; i < argc; ++i) for (int j = 1; j < i; ++j) {
        const char *a = strrchr(argv[i], '/'), *b = strrchr(argv[j], '/');
        a = a ? a+1 : argv[i]; b = b ? b+1 : argv[j];
        if (!strcmp(a,b)) { fprintf(stderr, "Duplicate image basename: %s\n", a); return 1; }
    }
    int failed = 0;
    for (int i = 1; i < argc; ++i) failed += process_image(argv[i]) != 0;
    printf("Images completed: %d | Failed: %d\n", argc-1-failed, failed);
    return failed ? 1 : 0;
}


## Compile

In [ ]:
%%bash
set -euo pipefail
gcc -O2 -DLODEPNG_NO_COMPILE_CPP -c lodepng.c -o lodepng.o
nvcc -O2 -lineinfo -DLODEPNG_NO_COMPILE_CPP sobel.cu lodepng.o -o sobel

## Run the main example

In [ ]:
%%bash
set -euo pipefail
./sobel ../data/images/flower.png | tee run.log


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
paths = ['../data/images/flower.png', 'flower_gx.png', 'flower_gy.png', 'flower_edges.png']
titles = ['Original image', 'Gradient in X direction', 'Gradient in Y direction', 'Sobel edge detection']
fig, axes = plt.subplots(2, 2, figsize=(10, 13))
for ax, path, title in zip(axes.flat, paths, titles):
    ax.imshow(Image.open(path).convert('RGB'))
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.savefig('flower_comparison.png', dpi=140)
plt.show()


## Correctness checks and measurements

The checks compare against independent CPU references and exercise invalid input. Five measured repeats follow a warm-up where appropriate. CPU timings cover the computation stated by each program; CUDA event timings cover kernels only, not file access, allocation or transfers. These scopes must not be compared as end-to-end speedups.

In [ ]:
%%bash
set -euo pipefail
python ../tests/verify.py 4 | tee validation.log

## Implementation summary

The two 3x3 Sobel masks are evaluated by a single CUDA kernel. Threads use linear pixel indices, with zero padding for neighbours outside the image.


## Output files

The archive contains the output generated by this run and its validation records.


In [ ]:
from pathlib import Path
import shutil, zipfile
from google.colab import files
archive = Path('/content/Task4_outputs.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for pattern in ['*.txt', '*.log', '*.png', 'validation/*.json']:
        for path in Path('.').glob(pattern):
            z.write(path, path.as_posix())
files.download(str(archive))